# Join Statements - Lab

## Introduction

In this lab, you'll practice your knowledge of `JOIN` statements, using various types of joins and various methods for specifying the links between them.

## Objectives

You will be able to:

* Write SQL queries that make use of various types of joins
* Compare and contrast the various types of joins
* Discuss how primary and foreign keys are used in SQL
* Decide and perform whichever type of join is best for retrieving desired data

## CRM ERD

In this lab, you'll use the same customer relationship management (CRM) database that you saw from the previous lesson.
<img src='https://curriculum-content.s3.amazonaws.com/data-science/images/Database-Schema.png' width="600">

## Connecting to the Database
Import the necessary packages and connect to the database `'data.sqlite'`.

In [13]:
# Your code here
import sqlite3
import pandas as pd

conn = sqlite3.connect("data.sqlite")

In [14]:
#Using Distinct function



## Select the names of all employees in Boston 

Hint: join the employees and offices tables. Select the first and last name.

In [15]:
q = """SELECT emp.firstName, emp.lastName FROM employees AS emp JOIN offices AS off  
ON emp.officeCode = off.officeCode WHERE city = 'Boston'
"""
pd.read_sql(q,conn)

,firstName,lastName
0,Julie,Firrelli
1,Steve,Patterson


## Are there any offices that have zero employees?
Hint: Combine the employees and offices tables and use a group by. Select the office code, city, and number of employees.

In [16]:
# Your code here
q = """SELECT off.officeCode, off.city, COUNT(emp.employeeNumber) AS NumberOfEmp 
FROM offices AS off
LEFT JOIN employees AS emp  ON 
emp.officeCode = off.officeCode 
GROUP BY off.officeCode, off.city
HAVING NumberOfEmp = 0 """
pd.read_sql(q,conn)

,officeCode,city,NumberOfEmp


## Write 3 questions of your own and answer them

In [17]:
# Answers will vary

# Example question: 
"""
How many customers are there per office?
"""

'\nHow many customers are there per office?\n'

In [18]:
"""
What are the names of all products sold?
"""
query = "SELECT DISTINCT productName FROM products;"
pd.read_sql(query, conn)


,productName
0,1969 Harley Davidson Ultimate Chopper
1,1952 Alpine Renault 1300
2,1996 Moto Guzzi 1100i
3,2003 Harley-Davidson Eagle Drag Bike
4,1972 Alfa Romeo GTA
...,...
105,The Titanic
106,The Queen Mary
107,American Airlines: MD-11S
108,Boeing X-32A JSF


In [19]:
"""
What is the TOP 10 most sold products?
"""

query = """
    SELECT products.productName, SUM(orderdetails.quantityOrdered) AS TotalQuantity
    FROM orderdetails
    JOIN products ON orderdetails.productCode = products.productCode
    GROUP BY products.productName
    ORDER BY TotalQuantity DESC
    LIMIT 10
"""
pd.read_sql(query, conn)


,productName,TotalQuantity
0,1992 Ferrari 360 Spider red,1808
1,1937 Lincoln Berline,1111
2,American Airlines: MD-11S,1085
3,1941 Chevrolet Special Deluxe Cabriolet,1076
4,1930 Buick Marquette Phaeton,1074
5,1940s Ford truck,1061
6,1969 Harley Davidson Ultimate Chopper,1057
7,1957 Chevy Pickup,1056
8,1964 Mercedes Tour Bus,1053
9,1956 Porsche 356A Coupe,1052


In [20]:
"""
How many customers are there?
"""
query = "SELECT COUNT(*) AS customerNumber FROM customers;"
pd.read_sql_query(query, conn)


,customerNumber
0,122


## Level Up 1: Display the names of every individual product that each employee has sold

Hint: You will need to use multiple `JOIN` clauses to connect all the way from employee names to product names.

In [21]:
query = """
    SELECT employees.firstName, employees.lastName, products.productName
    FROM employees
    JOIN customers ON customers.salesRepEmployeeNumber = employees.employeeNumber
    JOIN orders ON customers.customerNumber = orders.customerNumber
    JOIN orderDetails ON orders.orderNumber = orderdetails.orderNumber
    JOIN products ON orderdetails.productCode = products.productCode
"""
pd.read_sql(query, conn)

,firstName,lastName,productName
0,Leslie,Jennings,1958 Setra Bus
1,Leslie,Jennings,1940 Ford Pickup Truck
2,Leslie,Jennings,1939 Cadillac Limousine
3,Leslie,Jennings,1996 Peterbilt 379 Stake Bed with Outrigger
4,Leslie,Jennings,1968 Ford Mustang
...,...,...,...
2991,Martin,Gerard,1954 Greyhound Scenicruiser
2992,Martin,Gerard,1950's Chicago Surface Lines Streetcar
2993,Martin,Gerard,Diamond T620 Semi-Skirted Tanker
2994,Martin,Gerard,1911 Ford Town Car


## Level Up 2: Display the number of products each employee has sold

Alphabetize the results by employee last name.

Hint: Use the `quantityOrdered` column from `orderDetails`. Also, think about how to group the data when some employees might have the same first or last name.

In [22]:
query = """
SELECT employees.firstName, employees.lastName, products.productName, SUM(quantityOrdered) AS NumberOfProducts
FROM employees
  JOIN customers ON customers.salesRepEmployeeNumber = employees.employeeNumber
  JOIN orders ON customers.customerNumber = orders.customerNumber
  JOIN orderDetails ON orders.orderNumber = orderdetails.orderNumber
  JOIN products ON orderdetails.productCode = products.productCode
GROUP BY
  employees.firstName, employees.lastName
ORDER BY
  NumberOfProducts DESC
      
"""
pd.read_sql(query, conn)

,firstName,lastName,productName,NumberOfProducts
0,Gerard,Hernandez,1965 Aston Martin DB5,14231
1,Leslie,Jennings,1958 Setra Bus,11854
2,Pamela,Castillo,1972 Alfa Romeo GTA,9290
3,Larry,Bott,1972 Alfa Romeo GTA,8205
4,Barry,Jones,1952 Alpine Renault 1300,7486
5,George,Vanauf,1969 Harley Davidson Ultimate Chopper,7423
6,Peter,Marsh,1962 LanciaA Delta 16V,6632
7,Andy,Fixter,1996 Moto Guzzi 1100i,6246
8,Loui,Bondur,1952 Alpine Renault 1300,6186
9,Steve,Patterson,2001 Ferrari Enzo,5561


## Level Up 3: Display the names employees who have sold more than 200 different products

Hint: this is different from the previous question because the quantity sold doesn't matter, only the number of different products

In [23]:
#Discussed in class
query = """
SELECT emp.firstName, emp.lastName, COUNT (od.productCode) AS TotalProducts
FROM employees AS emp
JOIN customers AS cust 
  ON emp.employeeNumber = cust.salesRepEmployeeNumber
JOIN orders AS ord 
  USING(customerNumber)
JOIN orderdetails AS od 
  USING(orderNumber)
GROUP BY emp.firstName, emp.lastName
HAVING TotalProducts > 200  
"""
pd.read_sql(query, conn)

,firstName,lastName,TotalProducts
0,Barry,Jones,220
1,George,Vanauf,211
2,Gerard,Hernandez,396
3,Larry,Bott,236
4,Leslie,Jennings,331
5,Pamela,Castillo,272


## Summary

In [24]:
#Example self join

query = """
SELECT e.employeeNumber, m.employeeNumber, e.firstName, e.lastName, m.reportsTo
FROM employees e
  LEFT JOIN  employees m ON e.employeeNumber = m.employeeNumber
      
"""
pd.read_sql(query, conn)

,employeeNumber,employeeNumber,firstName,lastName,reportsTo
0,1002,1002,Diane,Murphy,
1,1056,1056,Mary,Patterson,1002
2,1076,1076,Jeff,Firrelli,1002
3,1088,1088,William,Patterson,1056
4,1102,1102,Gerard,Bondur,1056
5,1143,1143,Anthony,Bow,1056
6,1165,1165,Leslie,Jennings,1143
7,1166,1166,Leslie,Thompson,1143
8,1188,1188,Julie,Firrelli,1143
9,1216,1216,Steve,Patterson,1143


Congrats! You practiced using join statements and leveraged your foreign keys knowledge!